# Phase 8 — LoRA 파인튜닝

Notion 문서 기반 Q&A 데이터셋을 생성하고 Qwen2.5에 QLoRA 파인튜닝을 적용합니다.

## ⚙️ 셀 실행 환경 안내
| 표시 | 의미 |
|------|------|
| **✅ CPU 실행 가능** | 로컬 CPU 환경에서 준비 작업 |
| **⚠️ GPU 필수** | NVIDIA GPU + CUDA 필요. GPU 환경으로 이동 후 실행 |

## 권장 GPU 사양
- qwen2.5:3b 파인튜닝: VRAM 8GB 이상
- qwen2.5:7b 파인튜닝: VRAM 16GB 이상

## 파인튜닝 흐름
```
[CPU] 1. QA 데이터셋 자동 생성 (Claude API 활용)
[CPU] 2. Alpaca 형식 변환 및 저장
[CPU] 3. 패키지 설치 확인
[GPU] 4. Qwen2.5 모델 로드 + QLoRA 설정
[GPU] 5. 파인튜닝 실행
[GPU] 6. 튜닝 전/후 품질 비교
```

In [ ]:
# ─────────────────────────────────────────────
# ✅ CPU 실행 가능
# 셀 0: 환경 설정
# ─────────────────────────────────────────────
import sys, os, json, time
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

from dotenv import load_dotenv
load_dotenv('../.env')

# GPU 확인
try:
    import torch
    GPU_AVAILABLE = torch.cuda.is_available()
    if GPU_AVAILABLE:
        print(f'✅ GPU 감지: {torch.cuda.get_device_name(0)}')
        print(f'   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f}GB')
    else:
        print('⚠️  GPU 없음 — 셀 1~3만 실행 가능합니다.')
        print('   GPU 환경으로 이동 후 셀 4~6을 실행하세요.')
except ImportError:
    GPU_AVAILABLE = False
    print('⚠️  PyTorch 미설치 — 셀 1~3만 실행 가능합니다.')

In [ ]:
# ─────────────────────────────────────────────
# ✅ CPU 실행 가능
# 셀 1: Notion 문서 기반 Q&A 데이터셋 자동 생성
#
# Claude API로 Notion 벡터스토어의 청크를 읽어 Q&A 쌍 생성
# ─────────────────────────────────────────────
from langchain_anthropic import ChatAnthropic
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from src.embeddings import EmbeddingManager
from src.vectorstore import PostgresVectorStore

QA_GENERATION_PROMPT = """당신은 기술 문서 기반 Q&A 데이터셋 생성 전문가입니다.
아래 문서 청크를 읽고, 이 내용을 기반으로 Q&A 쌍 {n_pairs}개를 생성하세요.

요구사항:
1. 질문은 실제 사용자가 할 법한 자연스러운 질문이어야 합니다.
2. 답변은 문서 내용에 근거하여 정확하고 완전해야 합니다.
3. 다양한 유형의 질문을 포함하세요 (사실 확인, 방법 설명, 비교 등).

문서:
{document}

반드시 아래 JSON 배열 형식으로만 응답하세요 (다른 텍스트 없이):
[{{"question": "...", "answer": "..."}}]"""

qa_llm = ChatAnthropic(model='claude-3-haiku-20240307', temperature=0.5)
qa_chain = ChatPromptTemplate.from_template(QA_GENERATION_PROMPT) | qa_llm | JsonOutputParser()

print('✅ Q&A 생성 체인 초기화 완료 (Claude claude-3-haiku-20240307)')

In [ ]:
# ─────────────────────────────────────────────
# ✅ CPU 실행 가능
# 셀 2: 벡터스토어에서 문서 샘플링 후 Q&A 생성
# ─────────────────────────────────────────────
embedding_provider = 'openai' if os.getenv('OPENAI_API_KEY') else 'huggingface'
embeddings = EmbeddingManager(provider=embedding_provider).embeddings
vs = PostgresVectorStore(embeddings, collection_name='notion_docs')

# 다양한 주제를 커버하기 위해 여러 쿼리로 문서 샘플링
# 실제 Notion 내용에 맞는 키워드로 수정하세요
SAMPLE_QUERIES = [
    '프로젝트 개요',
    '기술 스택',
    '개발 프로세스',
    '트러블슈팅',
    '설계 원칙',
]
N_PAIRS_PER_DOC = 3   # 문서당 생성할 Q&A 쌍 수

all_qa_pairs = []
seen_contents = set()

for query in SAMPLE_QUERIES:
    docs = vs.similarity_search(query, k=2)
    for doc in docs:
        content_hash = hash(doc.page_content[:100])
        if content_hash in seen_contents:
            continue  # 중복 문서 건너뜀
        seen_contents.add(content_hash)

        title = doc.metadata.get('title', 'Unknown')
        print(f'  처리 중: {title[:40]}...')
        try:
            pairs = qa_chain.invoke({'document': doc.page_content[:1500], 'n_pairs': N_PAIRS_PER_DOC})
            for p in pairs:
                p['source'] = title
            all_qa_pairs.extend(pairs)
        except Exception as e:
            print(f'    ⚠️  생성 실패: {e}')

print(f'\n✅ Q&A 생성 완료: {len(all_qa_pairs)}개 쌍')

In [ ]:
# ─────────────────────────────────────────────
# ✅ CPU 실행 가능
# 셀 3: Alpaca 형식 변환 및 저장
# ─────────────────────────────────────────────
def to_alpaca(qa_pairs: list) -> list:
    """Q&A 쌍을 Alpaca 학습 형식으로 변환"""
    return [
        {
            'instruction': p['question'],
            'input': '',
            'output': p['answer']
        }
        for p in qa_pairs
    ]

alpaca_data = to_alpaca(all_qa_pairs)

# 저장
out_dir = Path('../data/processed')
out_dir.mkdir(parents=True, exist_ok=True)

dataset_path = out_dir / 'alpaca_dataset.json'
with open(dataset_path, 'w', encoding='utf-8') as f:
    json.dump(alpaca_data, f, ensure_ascii=False, indent=2)

# 원본 Q&A 쌍도 저장 (참고용)
raw_path = out_dir / 'qa_pairs_raw.json'
with open(raw_path, 'w', encoding='utf-8') as f:
    json.dump(all_qa_pairs, f, ensure_ascii=False, indent=2)

print(f'✅ Alpaca 데이터셋 저장: {dataset_path}')
print(f'   총 {len(alpaca_data)}개 샘플')
print(f'\n샘플 미리보기:')
print(json.dumps(alpaca_data[0], ensure_ascii=False, indent=2))

---
## ⚠️ GPU 필수 구간

아래 셀(4~6)은 NVIDIA GPU + CUDA 환경에서만 실행 가능합니다.

**GPU 환경 준비:**
```bash
# GPU 환경에서 의존성 설치
pip install transformers>=4.36.0 torch>=2.1.0 accelerate bitsandbytes>=0.43.0 peft>=0.10.0 datasets>=2.18.0 trl>=0.8.0

# data/processed/alpaca_dataset.json 파일을 GPU 환경으로 복사 후 실행
```

In [ ]:
# ─────────────────────────────────────────────
# ⚠️ GPU 필수
# 셀 4: 패키지 로드 및 QLoRA 설정
# ─────────────────────────────────────────────
if not GPU_AVAILABLE:
    print('⚠️  GPU 없음 — 이 셀은 GPU 환경에서만 실행하세요.')
    raise SystemExit(0)

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel
from datasets import load_dataset
from trl import SFTTrainer
import torch

MODEL_NAME = os.getenv('FINETUNE_BASE_MODEL', 'Qwen/Qwen2.5-3B-Instruct')

# QLoRA: 4bit 양자화 설정
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
)

# LoRA 어댑터 설정
lora_config = LoraConfig(
    r=8,                                     # rank (8 or 16 권장)
    lora_alpha=16,                            # alpha = r * 2
    target_modules=['q_proj', 'v_proj'],      # 어텐션 레이어
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
)

print(f'✅ QLoRA 설정 완료')
print(f'   기반 모델: {MODEL_NAME}')
print(f'   LoRA rank: {lora_config.r}, alpha: {lora_config.lora_alpha}')

In [ ]:
# ─────────────────────────────────────────────
# ⚠️ GPU 필수
# 셀 5: 모델 로드 및 학습 실행
# ─────────────────────────────────────────────
print(f'⏳ 모델 로딩: {MODEL_NAME} (첫 실행 시 다운로드에 시간이 걸립니다)')

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
)
model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, lora_config)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f'✅ 모델 로드 완료')
print(f'   학습 파라미터: {trainable:,} / {total:,} ({trainable/total*100:.2f}%)')

# 데이터셋 로드
dataset = load_dataset('json', data_files=str(dataset_path), split='train')
print(f'   학습 데이터: {len(dataset)}개 샘플')

def format_prompt(examples):
    texts = []
    for inst, inp, out in zip(examples['instruction'], examples['input'], examples['output']):
        body = f'### 추가 정보:\n{inp}\n\n' if inp else ''
        texts.append(f'### 질문:\n{inst}\n\n{body}### 답변:\n{out}')
    return texts

training_args = TrainingArguments(
    output_dir='../models/qwen-lora-adapter',
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    warmup_steps=10,
    logging_steps=10,
    save_steps=50,
    fp16=True,
    optim='paged_adamw_8bit',
    max_grad_norm=0.3,
    report_to='none',
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=lora_config,
    tokenizer=tokenizer,
    args=training_args,
    formatting_func=format_prompt,
    max_seq_length=512,
)

print('\n⏳ 학습 시작...')
trainer.train()

adapter_path = '../models/qwen-lora-adapter'
model.save_pretrained(adapter_path)
tokenizer.save_pretrained(adapter_path)
print(f'\n✅ 학습 완료. LoRA 어댑터 저장: {adapter_path}')

In [ ]:
# ─────────────────────────────────────────────
# ⚠️ GPU 필수
# 셀 6: 파인튜닝 전/후 품질 비교
# ─────────────────────────────────────────────
def generate(model, tokenizer, question, max_new_tokens=256):
    prompt = f'### 질문:\n{question}\n\n### 답변:\n'
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    text = tokenizer.decode(output[0], skip_special_tokens=True)
    return text.split('### 답변:')[-1].strip()

# 베이스 모델 로드 (비교용)
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, quantization_config=bnb_config, device_map='auto'
)
tuned_model = PeftModel.from_pretrained(base_model, '../models/qwen-lora-adapter')

eval_questions = [
    '이 지식베이스의 핵심 내용을 요약해주세요.',
    '가장 중요한 기술적 결정은 무엇인가요?',
]

comparison = []
for q in eval_questions:
    print(f'\n질문: {q}')
    print('─' * 60)
    base_ans  = generate(base_model,  tokenizer, q)
    tuned_ans = generate(tuned_model, tokenizer, q)
    print(f'[베이스]  {base_ans[:150]}...')
    print(f'[파인튜닝] {tuned_ans[:150]}...')
    comparison.append({'question': q, 'base': base_ans, 'tuned': tuned_ans})

# 결과 저장
comp_path = Path('../data/finetune_comparison.json')
with open(comp_path, 'w', encoding='utf-8') as f:
    json.dump(comparison, f, ensure_ascii=False, indent=2)
print(f'\n✅ 비교 결과 저장: {comp_path}')
print("""
다음 단계:
  - data/eval_dataset.json으로 evaluate_rag.py 재실행하여 정량 비교
  - python scripts/evaluate_rag.py
  - 어댑터를 LLMAdapter(provider='huggingface')로 연결하여 RAG 체인에 적용
""")